# Notebook 06 — Analysis of Notebook 05 outputs (multi-potential)

This notebook is the **analysis layer** for the current Notebook 05 workflow.

It is written to match job names created by the current validation notebook:

- `dev_<prefix>_<potid>`
- `prod_<prefix>_<potid>`

where `potid` can be any model identifier, for example:

- `MACE_A`, `MACE_B`, `MACE_C`, `MACE_D`
- `ACE_A`, `ACE_B`, ...
- `EAM_Mendelev_2019_CuZr`
- `2007_Mendelev-M-I_Cu-Zr_LAMMPS_ipr1`

## Main goals

1. Open the pyiron project and find Notebook 05 jobs.
2. Parse job names robustly for the current multi-potential workflow.
3. Reconstruct the main validation outputs:
   - smoke tests
   - EOS scans
   - amorphous / NCL-oriented workflow
   - vacancy jobs (if present)
4. Make analysis plots:
   - EOS curves
   - RDF
   - energy / temperature traces
5. Build compact summary tables for Paper 1.


## 0. Imports and user configuration

In [ ]:

# -----------------------------
# 0. Imports and user configuration
# -----------------------------
import os
import re
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyiron_atomistics import Project

PROJECT_PATH = "../cu_zr_mlip_project"

# If None, potentials are inferred automatically from parsed job names.
# You can also set this explicitly, for example:
# POTENTIAL_IDS = ["MACE_A", "MACE_B", "MACE_C", "MACE_D",
#                  "EAM_Mendelev_2019_CuZr", "2007_Mendelev-M-I_Cu-Zr_LAMMPS_ipr1"]
POTENTIAL_IDS = None

# Choose which run tag to analyze:
#   None     -> use both dev_ and prod_
#   "dev"    -> only dev_ jobs
#   "prod"   -> only prod_ jobs
RUN_TAG = None

# Notebook 05 EOS defaults
EOS_POINTS = 7
EOS_SCALES = np.linspace(0.96, 1.04, EOS_POINTS)

# RDF settings
RDF_RMAX = 8.0
RDF_BINS = 200

# Optional output folder
OUTPUT_DIR = "./notebook06_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 1. Open the pyiron project and load the job table

In [ ]:

pr = Project(PROJECT_PATH)
pr


In [ ]:

jt = pr.job_table()
print(jt.to_string())
print(f"Total jobs in project: {len(jt)}")

if "job" not in jt.columns:
    raise KeyError("The project job table does not contain a 'job' column.")

jt.head()


## 2. Parse current Notebook 05 job names

Notebook 05 currently creates job names through:

```python
jname(prefix, pot_spec) -> f"{run_tag}_{prefix}_{pot_spec['id']}"
```

So the parser below is written specifically for prefixes used in the current validation workflow.


In [ ]:

RUN_TAGS = ["dev", "prod"]

PATTERNS = [
    ("smoke_static", re.compile(r"^(?P<run_tag>dev|prod)_smoke_static_b2_(?P<potid>.+)$")),
    ("smoke_md", re.compile(r"^(?P<run_tag>dev|prod)_smoke_md_b2_(?P<potid>.+)$")),
    ("eos", re.compile(r"^(?P<run_tag>dev|prod)_eos_(?P<structure>FCC_Cu|HCP_Zr|B2_CuZr)_(?P<eos_idx>\d+)_(?P<potid>.+)$")),
    ("static_glass_seed", re.compile(r"^(?P<run_tag>dev|prod)_static_glass_seed_(?P<potid>.+)$")),
    ("melt", re.compile(r"^(?P<run_tag>dev|prod)_melt_(?P<potid>.+)$")),
    ("quench", re.compile(r"^(?P<run_tag>dev|prod)_quench_(?P<temperature>\d+)K_(?P<potid>.+)$")),
    ("glass_minimize", re.compile(r"^(?P<run_tag>dev|prod)_glass_minimize_(?P<potid>.+)$")),
    ("nve", re.compile(r"^(?P<run_tag>dev|prod)_nve_(?P<potid>.+)$")),
    ("glass_compressed", re.compile(r"^(?P<run_tag>dev|prod)_glass_compressed_(?P<potid>.+)$")),
    ("vac_perfect", re.compile(r"^(?P<run_tag>dev|prod)_vac_perfect_(?P<structure>FCC_Cu|HCP_Zr|B2_CuZr)_(?P<potid>.+)$")),
    ("vac_defect", re.compile(r"^(?P<run_tag>dev|prod)_vac_defect_(?P<structure>FCC_Cu|HCP_Zr|B2_CuZr)_(?P<potid>.+)$")),
]

def parse_notebook05_jobname(jobname: str):
    if not isinstance(jobname, str):
        return None
    for tag, pattern in PATTERNS:
        m = pattern.match(jobname)
        if m:
            out = {"tag": tag, "job": jobname}
            out.update(m.groupdict())
            if "eos_idx" in out and out["eos_idx"] is not None:
                out["eos_idx"] = int(out["eos_idx"])
            if "temperature" in out and out["temperature"] is not None:
                out["temperature"] = int(out["temperature"])
            return out
    return None

parsed_rows = []
for _, row in jt.iterrows():
    parsed = parse_notebook05_jobname(row["job"])
    if parsed is None:
        continue
    parsed["job_id"] = row["id"] if "id" in row.index else None
    parsed["status"] = row["status"] if "status" in row.index else None
    parsed_rows.append(parsed)

jt05 = pd.DataFrame(parsed_rows)

if RUN_TAG is not None:
    jt05 = jt05[jt05["run_tag"] == RUN_TAG].copy()

print(f"Detected Notebook 05 jobs: {len(jt05)}")
jt05.head()


In [ ]:

if len(jt05) == 0:
    raise RuntimeError(
        "No Notebook 05 jobs were detected. "
        "Check PROJECT_PATH or the job-name parser."
    )

if POTENTIAL_IDS is None:
    potids = sorted(jt05["potid"].dropna().unique().tolist())
else:
    potids = [p for p in POTENTIAL_IDS if p in set(jt05["potid"].dropna())]

print("Detected potentials:")
print(potids)

inventory_cols = [c for c in ["job", "status", "run_tag", "tag", "structure", "temperature", "eos_idx", "potid"] if c in jt05.columns]
display(jt05[inventory_cols].sort_values(["potid", "tag", "job"]).reset_index(drop=True))


## 3. Helper functions

In [ ]:

def row_to_job(row):
    if row is None:
        return None
    if isinstance(row, pd.Series):
        job_id = row.get("job_id", None)
    else:
        job_id = row
    if job_id is None:
        return None
    return pr.load(job_id)

def first_row(subdf, sort_cols=None):
    if len(subdf) == 0:
        return None
    if sort_cols is not None:
        subdf = subdf.sort_values(sort_cols)
    return subdf.iloc[0]

def find_rows(potid=None, tag=None, structure=None, run_tag=None):
    sub = jt05.copy()
    if potid is not None:
        sub = sub[sub["potid"] == potid]
    if tag is not None:
        sub = sub[sub["tag"] == tag]
    if structure is not None and "structure" in sub.columns:
        sub = sub[sub["structure"] == structure]
    if run_tag is not None:
        sub = sub[sub["run_tag"] == run_tag]
    return sub

def find_first_job(potid=None, tag=None, structure=None, run_tag=None, sort_cols=None):
    row = first_row(find_rows(potid=potid, tag=tag, structure=structure, run_tag=run_tag), sort_cols=sort_cols)
    if row is None:
        return None
    return row_to_job(row)

def safe_array(job, names):
    for name in names:
        try:
            arr = getattr(job.output, name)
            if arr is not None:
                return np.asarray(arr)
        except Exception:
            pass
    return None

def safe_last(job, names):
    arr = safe_array(job, names)
    if arr is None or len(arr) == 0:
        return np.nan
    return arr[-1]

def get_last_energy(job):
    return safe_last(job, ["energy_pot", "energy_tot"])

def get_last_temperature(job):
    return safe_last(job, ["temperature", "temperatures"])

def get_last_pressure(job):
    return safe_last(job, ["pressure", "press", "pressures"])

def get_final_structure(job):
    for step in [-1, None]:
        try:
            if step is None:
                s = job.get_structure()
            else:
                s = job.get_structure(iteration_step=step)
            if s is not None:
                return s
        except Exception:
            pass
    return None

def get_structure_trajectory(job):
    traj = []
    try:
        n = len(job.output.indices)
        for i in range(n):
            try:
                traj.append(job.get_structure(iteration_step=i))
            except Exception:
                pass
        if len(traj) > 0:
            return traj
    except Exception:
        pass

    try:
        if hasattr(job, "trajectory") and job.trajectory is not None:
            return list(job.trajectory())
    except Exception:
        pass

    return traj

def cell_volume(atoms):
    try:
        return float(atoms.get_volume())
    except Exception:
        try:
            return float(abs(np.linalg.det(np.asarray(atoms.cell))))
        except Exception:
            return np.nan

def structure_n_atoms(job):
    try:
        return len(job.structure)
    except Exception:
        s = get_final_structure(job)
        return len(s) if s is not None else np.nan


## 4. Project-level availability summary

In [ ]:

summary_rows = []

for potid in potids:
    row = {"potid": potid}
    for tag in ["smoke_static", "smoke_md", "melt", "glass_minimize", "nve", "glass_compressed"]:
        row[f"has_{tag}"] = len(find_rows(potid=potid, tag=tag)) > 0

    row["n_quench_jobs"] = len(find_rows(potid=potid, tag="quench"))
    row["n_eos_jobs"] = len(find_rows(potid=potid, tag="eos"))
    row["n_vacancy_jobs"] = len(find_rows(potid=potid, tag="vac_perfect")) + len(find_rows(potid=potid, tag="vac_defect"))

    j_smoke_static = find_first_job(potid=potid, tag="smoke_static")
    j_smoke_md = find_first_job(potid=potid, tag="smoke_md")
    j_melt = find_first_job(potid=potid, tag="melt")
    j_glass_min = find_first_job(potid=potid, tag="glass_minimize")
    j_nve = find_first_job(potid=potid, tag="nve")

    row["smoke_static_e_per_atom"] = (
        get_last_energy(j_smoke_static) / structure_n_atoms(j_smoke_static)
        if j_smoke_static is not None else np.nan
    )
    row["smoke_md_e_per_atom"] = (
        get_last_energy(j_smoke_md) / structure_n_atoms(j_smoke_md)
        if j_smoke_md is not None else np.nan
    )
    row["melt_T_last"] = get_last_temperature(j_melt) if j_melt is not None else np.nan
    row["glass_min_e_per_atom"] = (
        get_last_energy(j_glass_min) / structure_n_atoms(j_glass_min)
        if j_glass_min is not None else np.nan
    )
    row["nve_T_last"] = get_last_temperature(j_nve) if j_nve is not None else np.nan
    row["nve_E_last"] = get_last_energy(j_nve) if j_nve is not None else np.nan

    summary_rows.append(row)

availability_df = pd.DataFrame(summary_rows).sort_values("potid").reset_index(drop=True)
availability_df


## 5. EOS reconstruction from Notebook 05 jobs

In [ ]:

eos_rows = []

for potid in potids:
    sub = find_rows(potid=potid, tag="eos").sort_values(["structure", "eos_idx"])
    for _, row in sub.iterrows():
        job = row_to_job(row)
        e_last = get_last_energy(job)
        n_atoms = structure_n_atoms(job)
        eos_rows.append({
            "potid": potid,
            "structure": row["structure"],
            "eos_idx": row["eos_idx"],
            "scale": EOS_SCALES[row["eos_idx"]] if row["eos_idx"] < len(EOS_SCALES) else np.nan,
            "energy_last_eV": e_last,
            "energy_per_atom_eV": e_last / n_atoms if n_atoms and n_atoms > 0 else np.nan,
            "n_atoms": n_atoms,
        })

eos_df = pd.DataFrame(eos_rows)
eos_df.head()


In [ ]:

if len(eos_df) == 0:
    print("No EOS jobs found.")
else:
    display(eos_df.sort_values(["structure", "potid", "eos_idx"]).reset_index(drop=True))

    for structure_name in ["FCC_Cu", "HCP_Zr", "B2_CuZr"]:
        sub = eos_df[eos_df["structure"] == structure_name]
        if len(sub) == 0:
            continue

        plt.figure()
        for potid in sorted(sub["potid"].unique()):
            ss = sub[sub["potid"] == potid].sort_values("scale")
            plt.plot(ss["scale"], ss["energy_per_atom_eV"], marker="o", label=potid)
        plt.xlabel("Isotropic scale factor")
        plt.ylabel("Energy / atom (eV)")
        plt.title(f"EOS-style scan: {structure_name}")
        plt.legend()
        plt.show()


## 6. Vacancy formation summary (if vacancy jobs were run)

In [ ]:

vac_rows = []

for potid in potids:
    for structure_name in ["FCC_Cu", "HCP_Zr", "B2_CuZr"]:
        jperf = find_first_job(potid=potid, tag="vac_perfect", structure=structure_name)
        jdef = find_first_job(potid=potid, tag="vac_defect", structure=structure_name)
        if jperf is None or jdef is None:
            continue

        e_perf = get_last_energy(jperf)
        e_def = get_last_energy(jdef)
        n_perf = structure_n_atoms(jperf)

        if n_perf is None or n_perf <= 1:
            e_form = np.nan
        else:
            e_form = e_def - ((n_perf - 1) / n_perf) * e_perf

        vac_rows.append({
            "potid": potid,
            "structure": structure_name,
            "n_perf": n_perf,
            "E_perf_eV": e_perf,
            "E_def_eV": e_def,
            "E_form_eV": e_form,
        })

vac_df = pd.DataFrame(vac_rows)
vac_df


## 7. Representative amorphous jobs for structural and trace analysis

In [ ]:

def choose_representative_glass_job(potid):
    # Prefer minimized glass, then nve, then final quench
    j = find_first_job(potid=potid, tag="glass_minimize")
    if j is not None:
        return j, "glass_minimize"

    j = find_first_job(potid=potid, tag="nve")
    if j is not None:
        return j, "nve"

    sub = find_rows(potid=potid, tag="quench").sort_values("temperature")
    row = first_row(sub)
    if row is not None:
        return row_to_job(row), f"quench_{int(row['temperature'])}K"

    return None, None

rep_rows = []
rep_jobs = {}

for potid in potids:
    job, source = choose_representative_glass_job(potid)
    rep_jobs[potid] = job
    rep_rows.append({"potid": potid, "representative_source": source, "has_job": job is not None})

rep_df = pd.DataFrame(rep_rows)
rep_df


## 8. RDF analysis

In [ ]:

from ase.geometry.analysis import Analysis

rdf_curves = {}

for potid, job in rep_jobs.items():
    if job is None:
        continue
    atoms = get_final_structure(job)
    if atoms is None:
        continue
    try:
        analysis = Analysis(atoms)
        r, g_r = analysis.get_rdf(rmax=RDF_RMAX, nbins=RDF_BINS)
        rdf_curves[potid] = (np.asarray(r), np.asarray(g_r))
    except Exception as exc:
        print(f"Skipping RDF for {potid}: {exc}")

plt.figure()
for potid, (r, g_r) in rdf_curves.items():
    plt.plot(r, g_r, label=potid)
plt.xlabel("r (Å)")
plt.ylabel("g(r)")
plt.title("Amorphous RDF comparison")
plt.legend()
plt.show()


## 9. Energy and temperature traces

In [ ]:

def plot_trace(job_lookup, title, ylabels):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    found_e = False
    found_t = False

    for potid, job in job_lookup.items():
        if job is None:
            continue

        e = safe_array(job, ["energy_pot", "energy_tot"])
        t = safe_array(job, ["temperature", "temperatures"])

        if e is not None:
            axes[0].plot(e, label=potid)
            found_e = True
        if t is not None:
            axes[1].plot(t, label=potid)
            found_t = True

    axes[0].set_title(f"{title} — energy")
    axes[0].set_xlabel("Step index")
    axes[0].set_ylabel(ylabels[0])

    axes[1].set_title(f"{title} — temperature")
    axes[1].set_xlabel("Step index")
    axes[1].set_ylabel(ylabels[1])

    if found_e:
        axes[0].legend()
    if found_t:
        axes[1].legend()

    plt.tight_layout()
    plt.show()

melt_jobs = {potid: find_first_job(potid=potid, tag="melt") for potid in potids}
nve_jobs = {potid: find_first_job(potid=potid, tag="nve") for potid in potids}

plot_trace(melt_jobs, title="Melt stage", ylabels=("Energy (eV)", "Temperature (K)"))
plot_trace(nve_jobs, title="NVE stage", ylabels=("Energy (eV)", "Temperature (K)"))


## 10. Compact NCL-oriented workflow summary

In [ ]:

final_rows = []

for potid in potids:
    row = {"potid": potid}

    j_static = find_first_job(potid=potid, tag="static_glass_seed")
    j_melt = find_first_job(potid=potid, tag="melt")
    j_glass_min = find_first_job(potid=potid, tag="glass_minimize")
    j_nve = find_first_job(potid=potid, tag="nve")
    j_comp = find_first_job(potid=potid, tag="glass_compressed")

    row["glass_seed_e_per_atom"] = (
        get_last_energy(j_static) / structure_n_atoms(j_static)
        if j_static is not None else np.nan
    )
    row["melt_T_last"] = get_last_temperature(j_melt) if j_melt is not None else np.nan
    row["glass_min_e_per_atom"] = (
        get_last_energy(j_glass_min) / structure_n_atoms(j_glass_min)
        if j_glass_min is not None else np.nan
    )
    row["glass_compressed_e_per_atom"] = (
        get_last_energy(j_comp) / structure_n_atoms(j_comp)
        if j_comp is not None else np.nan
    )

    if j_nve is not None:
        e_nve = safe_array(j_nve, ["energy_tot", "energy_pot"])
        row["nve_energy_start_eV"] = e_nve[0] if e_nve is not None and len(e_nve) > 0 else np.nan
        row["nve_energy_end_eV"] = e_nve[-1] if e_nve is not None and len(e_nve) > 0 else np.nan
        row["nve_energy_drift_eV"] = (
            (e_nve[-1] - e_nve[0]) if e_nve is not None and len(e_nve) > 1 else np.nan
        )
        row["nve_T_last"] = get_last_temperature(j_nve)
    else:
        row["nve_energy_start_eV"] = np.nan
        row["nve_energy_end_eV"] = np.nan
        row["nve_energy_drift_eV"] = np.nan
        row["nve_T_last"] = np.nan

    if potid in rdf_curves:
        r, g_r = rdf_curves[potid]
        row["rdf_first_peak_r_A"] = float(r[np.argmax(g_r)])
        row["rdf_first_peak_height"] = float(np.max(g_r))
    else:
        row["rdf_first_peak_r_A"] = np.nan
        row["rdf_first_peak_height"] = np.nan

    if len(vac_df) > 0:
        sub_v = vac_df[vac_df["potid"] == potid]
        row["vac_FCC_Cu_eV"] = sub_v.loc[sub_v["structure"] == "FCC_Cu", "E_form_eV"].iloc[0] if np.any(sub_v["structure"] == "FCC_Cu") else np.nan
        row["vac_HCP_Zr_eV"] = sub_v.loc[sub_v["structure"] == "HCP_Zr", "E_form_eV"].iloc[0] if np.any(sub_v["structure"] == "HCP_Zr") else np.nan
    else:
        row["vac_FCC_Cu_eV"] = np.nan
        row["vac_HCP_Zr_eV"] = np.nan

    final_rows.append(row)

final_df = pd.DataFrame(final_rows).sort_values("potid").reset_index(drop=True)
final_df


## 11. Save analysis tables

This writes the main tables to disk so they can be reused in paper-writing scripts or later notebooks.


In [ ]:

availability_df.to_csv(os.path.join(OUTPUT_DIR, "availability_df.csv"), index=False)
eos_df.to_csv(os.path.join(OUTPUT_DIR, "eos_df.csv"), index=False)
vac_df.to_csv(os.path.join(OUTPUT_DIR, "vacancy_df.csv"), index=False)
rep_df.to_csv(os.path.join(OUTPUT_DIR, "representative_jobs_df.csv"), index=False)
final_df.to_csv(os.path.join(OUTPUT_DIR, "final_df.csv"), index=False)

print(f"Saved analysis tables to: {OUTPUT_DIR}")


## 12. Notes

This Notebook 06 is synchronized to the current multi-potential Notebook 05 design.

It assumes the job names are created with:

```python
f"{run_tag}_{prefix}_{pot_spec['id']}"
```

So if Notebook 05 naming changes later, the parser in this notebook should be updated accordingly.
